# Create Villum Awards (Villum Fonden, Denmark)

Creates Villum Fonden awards from the public project database published on veluxfonden.dk.

**Prerequisites:** run `scripts/local/villum_to_s3.py` to harvest + upload first.

**Data source:** `https://veluxfonden.dk/en/data.js` — a single JavaScript asset wrapping `siteData.merge(JSON.parse('<one big JSON blob>'))`. The blob contains the full Velux/Villum grant database (15,597 projects in the current source pull). We filter to **foundation_id=59 (Villum Foundation)** via the `grant_sub_area -> grant_area -> foundation` chain (2,917 Villum projects in the current source pull), then keep named flagship research schemes only for this clean pilot ingest: **980 awards, 2011-2025**. F4320310489 (Velux Fonden, welfare/culture/heritage) is a sibling ingest under a different funder_id from the same file.

**S3 location:** `s3a://openalex-ingest/awards/villum/villum_projects.parquet`

**Villum funder in OpenAlex:** funder_id 4320310490 · display_name "Villum Fonden" · DK.

**Input columns from villum_to_s3.py:**
- `funder_award_id` = `villum-{project_id}` (Villum has no public cited grant prefix; the project_id from data.js is URL-stable).
- `title` (English), `amount` (DKK, 0/neg already NULLed §6.7), `currency` = 'DKK', `year`.
- `pi_name`, `pi_given_name`, `pi_family_name` — split from a single `pa_name` field; §6.4a institution-as-PI guard NULLs the split when name contains org tokens.
- `institution_name`, `institution_country` (dereferenced from explicit organisation/country FK lookup tables in data.js; not inferred).
- `funder_scheme` = grant_sub_area.title.
- `funder_area` = grant_area.title.
- **Demographic PII (`pa_age`, `pa_gender`) is intentionally dropped at scrape time** — the foundation publishes it for transparency, but we don't propagate it into the awards table.

**Notes:**
- 980 clean flagship research rows after the named-research-scheme filter.
- PI family-name coverage 99.8% (978/980); institution coverage 100%.
- DKK amount coverage 100%; median DKK 2,499,160; max DKK 39,999,999.
- Year range: 2011-2025.
- Dropped fellowship/other-research/non-research rows should be evaluated in a follow-up widen pass rather than mixed into this pilot.

provenance `villum_veluxfonden`, priority 191.

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.villum_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/villum/villum_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.villum_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.villum_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.villum_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

Must return exactly 1 row. If 0, STOP — the funder is missing from `openalex.common.funder`.

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Villum Fonden funder_id 4320310490 missing or duplicated in openalex.common.funder'
) AS villum_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320310490;

## Step 2: Create Villum Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.villum_awards
USING delta
AS
WITH
villum_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320310490  -- Villum Fonden
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) ELSE NULL END as amount,
        CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'DKK' ELSE NULL END as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN LOWER(g.funder_scheme) RLIKE '(visiting professor|young invest|kavli|postdoc|fellowship|talent|phd|ph\\.d|stipend)' THEN 'fellowship'
            WHEN LOWER(g.funder_area) RLIKE '(technical|scientific|research)' THEN 'research'
            ELSE 'grant'
        END as funding_type,
        g.funder_scheme as funder_scheme,
        'villum_veluxfonden' as provenance,
        CASE WHEN TRY_CAST(g.year AS INT) IS NOT NULL
                  AND TRY_CAST(g.year AS INT) <= YEAR(current_date()) + 1
             THEN CAST(CONCAT(g.year, '-01-01') AS DATE) ELSE NULL END as start_date,
        CAST(NULL AS DATE) as end_date,
        CASE WHEN TRY_CAST(g.year AS INT) <= YEAR(current_date()) + 1
             THEN TRY_CAST(g.year AS INT) ELSE NULL END as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL OR g.institution_name IS NOT NULL THEN
                struct(
                    g.pi_given_name as given_name,
                    g.pi_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution_name as name,
                        g.institution_country as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        COALESCE(g.landing_url, 'https://veluxfonden.dk/en/basic-page/projects-granted') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.villum_raw g
    CROSS JOIN villum_funder f
    WHERE g.funder_award_id IS NOT NULL
      AND TRIM(CAST(g.funder_award_id AS STRING)) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'villum_veluxfonden' AND priority = 191;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    191 as priority
FROM openalex.awards.villum_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_villum_awards FROM openalex.awards.villum_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, funding_type, amount, currency, start_year, lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.villum_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a frequency check on PI names + display_names (catches systematic scraper bugs even when §6.3 reports 100% coverage)
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.villum_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n FROM openalex.awards.villum_awards GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.villum_awards GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt FROM openalex.awards.villum_awards WHERE funder_scheme IS NOT NULL GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
-- §6.3 completeness. Expected: 980 total, 100% title/amount/institution/start_date, 99.8% PI family-name coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) as pct_with_pi
FROM openalex.awards.villum_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.villum_awards WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.villum_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY lead_investigator.affiliation.name ORDER BY grant_count DESC LIMIT 20;

## §6.5: Net-New Survivor Estimate (pre-fold)

In [ ]:
%sql
-- Informational Checkpoint-B query: rows expected to survive lower-priority dedup.
SELECT COUNT(*) AS net_new_survivors
FROM openalex.awards.villum_awards a
WHERE NOT EXISTS (
    SELECT 1
    FROM openalex.awards.openalex_awards_raw x
    WHERE x.id = a.id
      AND x.priority < 191
);